# Lab type: debug
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: K-Means Clustering
# Task: Find and fix the 3 bugs in the k-means pipeline below. After fixing each bug, write a one-sentence explanation in the comment cell below it.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt


## Step 1: Load and Prepare Data


In [ ]:
# Create synthetic customer data
np.random.seed(42)

# Generate three natural clusters
cluster_1 = np.random.normal([5, 5], 1, (50, 2))
cluster_2 = np.random.normal([15, 15], 1.5, (50, 2))
cluster_3 = np.random.normal([5, 15], 1, (40, 2))

X = np.vstack([cluster_1, cluster_2, cluster_3])
print(f"Data shape: {X.shape}")


## Step 2: Bug 1 — Missing Scaling


In [ ]:
# BUG 1: Scaling is skipped entirely
# (If your features have different ranges, k-means will weight the largest-range feature too heavily)
X_processed = X  # Missing: StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_processed)

print(f"Cluster assignments: {np.unique(labels)}")


**Fix explanation here:**


<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug is:** `X_processed = X` passes raw, unscaled features directly to KMeans.

**Why it causes wrong behaviour:** K-means minimises inertia (sum of squared Euclidean distances). When features have very different numerical ranges — for example, `annual_spend` in the thousands vs. `age` in the tens — the larger-range feature dominates the distance calculation. The algorithm effectively clusters on that one feature alone and ignores everything else.

**How the fix resolves it:** `scaler = StandardScaler(); X_processed = scaler.fit_transform(X)` transforms all features to zero mean and unit variance. Every feature then contributes equally to distance, so the clustering reflects the full structure of the data rather than the scale of individual columns.

</details>

## Step 3: Bug 2 — Fitting Scaler on Full Data (Data Leakage)


In [ ]:
# BUG 2: Scaler is fit on all the data before clustering
# (In production, you'd fit on training data, then transform test data with those same parameters)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # BUG: Fitting on full data introduces leakage

kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_scaled)

print(f"Cluster assignments with scaling: {np.unique(labels, return_counts=True)}")


**Fix explanation here:**


<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug is:** `scaler.fit_transform(X)` fits the scaler on the entire dataset, including what would be test or future data in a real pipeline.

**Why it causes wrong behaviour:** The scaler's mean and standard deviation are computed using data the model should never have seen at fit time. In production, incoming data is transformed with these "contaminated" statistics — the scaling parameters reflect global dataset properties, not just the training distribution — which can cause subtle shifts in cluster assignments for new data.

**How the fix resolves it:** Fit the scaler only on training data (`scaler.fit(X_train)`), then apply `scaler.transform()` to both training and any new data. This ensures the same fixed parameters are used to transform all future observations, reproducing the training-time feature space consistently.

</details>

## Step 4: Bug 3 — Hardcoded Number of Clusters


In [ ]:
# BUG 3: n_clusters is hardcoded to 3
# (You should validate k with the elbow method, silhouette score, or domain knowledge)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)  # BUG: No justification for k=3
labels = kmeans.fit_predict(X_scaled)

print(f"Inertia: {kmeans.inertia_:.2f}")
print(f"Silhouette score: {kmeans.score(X_scaled):.2f}")  # Lower is better for score metric


**Fix explanation here:**


<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug is:** `n_clusters=3` is hardcoded with no validation that k=3 is actually the right number of clusters.

**Why it causes wrong behaviour:** Without checking the elbow curve or silhouette scores, k=3 may over- or under-partition the data. Additionally, `kmeans.score(X_scaled)` returns the *negative inertia*, so the comment "lower is better" is misleading — a less-negative score (closer to 0) means worse fit, not better.

**How the fix resolves it:** Sweep over a range of k values, plot inertia (elbow method) and silhouette scores, and choose k where the elbow occurs *and* silhouette is highest. Use `silhouette_score(X_scaled, labels)` from `sklearn.metrics` instead of `kmeans.score()` to get an interpretable metric.

</details>

## Step 5: Corrected Pipeline


In [ ]:
# After fixing all three bugs, the corrected pipeline should:
# 1. Scale the data consistently
# 2. Avoid data leakage by fitting scaler appropriately
# 3. Justify the choice of k with validation metrics
# Write your corrected code here


<details>
<summary>🔑 Reveal summary answers</summary>

1. **Missing scaling:** Always StandardScale before k-means — unscaled features give outsized weight to large-range dimensions, effectively discarding all others.
2. **Data leakage:** Fit the scaler on training data only; transform all other data with those fixed parameters.
3. **Hardcoded k:** Validate k with the elbow method and silhouette score — never choose k without evidence, and use `silhouette_score()` rather than `kmeans.score()` for interpretable quality measurement.

</details>

## Summary

K-means is simple, but the devil is in the details:
- Scale your data consistently
- Choose k with evidence, not guesswork
- Understand what k-means can and cannot do (it assumes spherical clusters)

**Next lesson:** Choosing k — rigorous methods for validating your cluster count.
